# Day 6 — LSTM Forecasting with PyTorch Lightning
Trains a stacked LSTM on daily revenue, evaluates against the test set, and compares with the Day 5 Prophet baseline.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src.forecasting import (
    train_test_split_ts,
    evaluate_forecast,
    load_prophet,
    get_prophet_test_preds,
)
from src.lstm_lightning import (
    RetailDataModule,
    LSTMLightning,
    build_trainer,
    train_lstm_lightning,
    forecast_lstm_lightning,
)

os.makedirs('models', exist_ok=True)
os.makedirs('reports/figures', exist_ok=True)

## 1. Load Data

In [ ]:
df = pd.read_csv('data/processed/daily_revenue_ts.csv', index_col=0, parse_dates=True)
train, test = train_test_split_ts(df)

print(f'Total rows : {len(df)}')
print(f'Train rows : {len(train)}  ({train.index.min().date()} -> {train.index.max().date()})')
print(f'Test rows  : {len(test)}   ({test.index.min().date()} -> {test.index.max().date()})')
df['Revenue'].plot(figsize=(14, 3), title='Daily Revenue — full series', color='steelblue')
plt.tight_layout()
plt.show()

## 2. DataModule — inspect batch shapes

In [ ]:
SEQ_LEN    = 30
BATCH_SIZE = 32

dm = RetailDataModule(train['Revenue'], seq_len=SEQ_LEN, batch_size=BATCH_SIZE, val_split=0.1)
dm.setup()

x_batch, y_batch = next(iter(dm.train_dataloader()))
print(f'X batch shape : {tuple(x_batch.shape)}  — (batch, seq_len, features)')
print(f'y batch shape : {tuple(y_batch.shape)}  — (batch, 1)')
print(f'Train batches : {len(dm.train_dataloader())}')
print(f'Val   batches : {len(dm.val_dataloader())}')

## 3. Train

In [ ]:
model, dm = train_lstm_lightning(
    train['Revenue'],
    seq_len=SEQ_LEN,
    hidden_size=64,
    num_layers=2,
    dropout=0.2,
    lr=1e-3,
    batch_size=BATCH_SIZE,
    max_epochs=50,
    patience=10,
)

## 4. Training Curve — train_loss vs val_loss

In [ ]:
# Lightning logs are stored in the MLflow run; we read them back via the trainer callback metrics.
# For a quick in-notebook view, capture metrics manually via a custom approach:
# Re-run a short training with a MetricsCallback to capture epoch losses.

import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping

class MetricsHistory(pl.Callback):
    def __init__(self):
        self.train_loss = []
        self.val_loss = []
    def on_train_epoch_end(self, trainer, pl_module):
        self.train_loss.append(trainer.callback_metrics.get('train_loss', float('nan')))
    def on_validation_epoch_end(self, trainer, pl_module):
        self.val_loss.append(trainer.callback_metrics.get('val_loss', float('nan')))

history = MetricsHistory()
dm2 = RetailDataModule(train['Revenue'], seq_len=SEQ_LEN, batch_size=BATCH_SIZE, val_split=0.1)
m2  = LSTMLightning(hidden_size=64, num_layers=2, dropout=0.2, lr=1e-3)
t2  = pl.Trainer(
    max_epochs=50,
    accelerator='auto',
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, mode='min'), history],
    logger=False,
    enable_progress_bar=True,
)
t2.fit(m2, datamodule=dm2)

epochs = range(1, len(history.val_loss) + 1)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(epochs, [float(v) for v in history.train_loss[:len(epochs)]], label='train_loss', color='steelblue')
ax.plot(epochs, [float(v) for v in history.val_loss],                 label='val_loss',   color='darkorange')
ax.set_title('LSTM Training Curve — Loss per Epoch', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('reports/figures/lstm_training_curve.png', dpi=150)
plt.show()
print('saved reports/figures/lstm_training_curve.png')

## 5. Forecast on Test Set

In [ ]:
lstm_preds = forecast_lstm_lightning(
    model,
    seed_series=train['Revenue'],
    steps=len(test),
    scaler=dm.scaler,
    seq_len=SEQ_LEN,
)

metrics = evaluate_forecast(test['Revenue'], lstm_preds)
print(f'MAE  : {metrics["mae"]}')
print(f'RMSE : {metrics["rmse"]}')
print(f'MAPE : {metrics["mape"]}%')

## 6. Forecast Comparison — Actual vs Prophet vs LSTM-Lightning

In [ ]:
try:
    prophet_model = load_prophet()
    prophet_preds = get_prophet_test_preds(prophet_model, test, target='Revenue')
    has_prophet = True
except FileNotFoundError:
    print('Prophet model not found — plot will show LSTM only. Run run_models.py first.')
    has_prophet = False

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test.index, test['Revenue'].values, label='Actual',         color='steelblue',  linewidth=1.2)
ax.plot(test.index, lstm_preds,             label='LSTM-Lightning', color='seagreen',   linestyle='--', linewidth=1.2)
if has_prophet:
    ax.plot(test.index, prophet_preds,      label='Prophet',        color='darkorange', linestyle='--', linewidth=1.2)

ax.set_title('Forecast Comparison — Test Period', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Revenue (£)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('reports/figures/lstm_lightning_forecast.png', dpi=150)
plt.show()
print('saved reports/figures/lstm_lightning_forecast.png')

## 7. MLflow Run Summary

In [ ]:
import mlflow

mlflow.set_experiment('retailpulse-forecasting')
runs = mlflow.search_runs(experiment_names=['retailpulse-forecasting'])

cols = ['tags.mlflow.runName', 'metrics.mae', 'metrics.rmse', 'metrics.mape']
available = [c for c in cols if c in runs.columns]
print(runs[available].dropna(subset=['metrics.mae']).to_string(index=False))